In [2]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [3]:
#%load_ext pyinstrument

In [4]:
USE_NEGATIVE_WGT = False
MAX_ITER = 1

In [5]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-09-08-02_29_45_PM'

In [6]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [7]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [8]:
experiment_config = {
     "experiment": {
        "model": "hill_climb",
        "exp_oof_path": "2026-09-07-07_15_19_PM_optuna_xgboost",
        "description": "xgb hill climb sample",
        "use_negative_wgt": USE_NEGATIVE_WGT,
        "max_iter": MAX_ITER
    }
}
experiment_config["experiment"]["id"] = f"{dt_str}_{experiment_config["experiment"]["model"]}"

In [9]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
y = raw_train_df[target_column]

In [10]:
exp_oof_path = Path(output_path) / "experiments" / experiment_config["experiment"]["exp_oof_path"]
tune_trials_df = pd.read_csv(exp_oof_path / "optuna_trials.csv")

trial_names = []
oof_dfs = pd.DataFrame()
ss_dfs = pd.DataFrame()
oof_id = None
submission_id = None

for subdir in exp_oof_path.iterdir():
    if not subdir.is_dir():
        continue

    trial_names.append(subdir.name)
    oof_file = subdir / "oof.csv"

    if oof_file.exists():
        file_df = pd.read_csv(oof_file)
        if oof_id is None:
            oof_id = file_df['id']
        oof_dfs[subdir.name] = file_df[target_column]

    submission_file = subdir / "submission.csv"
    
    if submission_file.exists():
        file_df = pd.read_csv(submission_file)
        if submission_id is None:
            submission_id = file_df['id']
        ss_dfs[subdir.name] = file_df[target_column]

y_valid = y.iloc[oof_id]
oof_dfs

,xgboost_trial_0,xgboost_trial_1,xgboost_trial_10,xgboost_trial_11,xgboost_trial_12,xgboost_trial_13,xgboost_trial_14,xgboost_trial_15,xgboost_trial_16,xgboost_trial_17,...,xgboost_trial_45,xgboost_trial_46,xgboost_trial_47,xgboost_trial_48,xgboost_trial_49,xgboost_trial_5,xgboost_trial_6,xgboost_trial_7,xgboost_trial_8,xgboost_trial_9
0,0.305121,0.215241,0.323119,0.524296,0.184538,0.371715,0.325952,0.408845,0.199777,0.387923,...,0.266145,0.389268,0.385583,0.279230,0.186753,0.210465,0.190994,0.230297,0.234677,0.399762
1,0.949019,0.996324,0.994965,0.979946,0.990566,0.970919,0.926197,0.900430,0.994565,0.933267,...,0.995024,0.856108,0.958407,0.977938,0.991827,0.985588,0.988359,0.980854,0.983707,0.928011
2,0.999957,0.999924,0.999567,0.999978,0.999988,0.999915,0.999376,0.999774,0.999973,0.994034,...,0.999969,0.992509,0.999908,0.999911,0.999912,0.999630,0.999916,0.999556,0.999818,0.999191
3,0.998327,0.999822,0.996553,0.998345,0.999685,0.996640,0.987900,0.996018,0.999410,0.974082,...,0.998190,0.920307,0.996108,0.999703,0.996787,0.997587,0.999216,0.995593,0.998546,0.991254
4,0.397003,0.392243,0.676195,0.455615,0.464484,0.460544,0.475495,0.587523,0.473708,0.499477,...,0.447886,0.498726,0.549707,0.420104,0.371356,0.569976,0.462955,0.446375,0.458753,0.476871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138269,0.999833,0.999661,0.998900,0.999828,0.999804,0.999015,0.997152,0.999341,0.999112,0.988941,...,0.999279,0.961670,0.999333,0.999511,0.998925,0.998952,0.997471,0.990656,0.998104,0.997078
138270,0.996711,0.999453,0.996557,0.997059,0.994342,0.992704,0.993622,0.997904,0.989250,0.987784,...,0.997568,0.972730,0.997045,0.998246,0.992116,0.997919,0.994361,0.981294,0.997064,0.995839
138271,0.075180,0.108483,0.128533,0.096514,0.048379,0.104085,0.193224,0.038707,0.048598,0.202226,...,0.052392,0.276429,0.043300,0.072390,0.031468,0.055186,0.068306,0.073351,0.071335,0.183412
138272,0.289177,0.349518,0.121804,0.137183,0.163197,0.170044,0.275086,0.332171,0.195628,0.241637,...,0.261938,0.536129,0.206190,0.163302,0.307132,0.553618,0.236980,0.383474,0.233200,0.276676


In [11]:
best_start_model_idx = tune_trials_df['value'].idxmax()
ensemble_score = tune_trials_df.iloc[best_start_model_idx]['value']

In [12]:
start = -0.50
if not USE_NEGATIVE_WGT: 
    start = 0.01

weights = np.arange(start, 0.51, 0.01)
weights.shape

(50,)

In [13]:
y_valid_np = np.asarray(y_valid)
pos_mask = y_valid_np == 1

n_pos = pos_mask.sum()
n_neg = len(y_valid_np) - n_pos

def fast_auc_batch(y_true, predictions):
    ranks = rankdata(predictions, axis=0, method="average")
    pos_rank_sum = ranks[y_true == 1].sum(axis=0)
    auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

    return auc

In [14]:
iter = 0

models = [f'xgboost_trial_{best_start_model_idx}']
wgts = []
scores = [ensemble_score]

ensemble_oof = oof_dfs[models[0]].to_numpy().reshape(-1, 1)
submission_oof = ss_dfs[models[0]].to_numpy().reshape(-1, 1)

start_time = time.time()

while iter < MAX_ITER:
    model_valid_scores = []
    best_weight_idxs = []

    for model in oof_dfs.columns:
        model_oof = oof_dfs[model].to_numpy().reshape(-1, 1)

        weight_oofs = (
            model_oof * weights
            + ensemble_oof * (1 - weights)
        )

        weight_scores = fast_auc_batch(
            y_valid_np,
            weight_oofs
        )

        max_idx = np.argmax(weight_scores)
        max_weight_score = weight_scores[max_idx]

        model_valid_scores.append(max_weight_score)
        best_weight_idxs.append(max_idx)

    model_valid_scores = np.array(model_valid_scores)

    max_model_idx = np.argmax(model_valid_scores)
    max_weight_idx = best_weight_idxs[max_model_idx]

    candidate_score = model_valid_scores[max_model_idx]

    model_name = oof_dfs.columns[max_model_idx]
    candidate_weight = weights[max_weight_idx]

    candidate_oof = oof_dfs[model_name].to_numpy().reshape(-1, 1)
    candidate_ss = ss_dfs[model_name].to_numpy().reshape(-1, 1)

    if candidate_score <= ensemble_score:
        break

    ensemble_oof = (
        candidate_oof * candidate_weight
        + ensemble_oof * (1 - candidate_weight)
    )

    submission_oof = (
        candidate_ss * candidate_weight
        + submission_oof * (1 - candidate_weight)
    )

    ensemble_score = candidate_score

    models.append(model_name)
    wgts.append(candidate_weight)
    scores.append(candidate_score)

    iter += 1

    wgt = np.array([1.0])

    for w in wgts:
        wgt = wgt * (1 - w)
        wgt = np.concatenate([wgt, np.array([w])])

    model_weight_df = pd.DataFrame({
        'model': models,
        'weight': wgt,
        'ensemble_score': scores
    })
    print(model_weight_df)

elapsed = time.time() - start_time

              model  weight  ensemble_score
0  xgboost_trial_25    0.66        0.967550
1  xgboost_trial_49    0.34        0.967735


In [18]:
oof_df = pd.DataFrame({'id': oof_id, target_column: ensemble_oof.reshape(-1)})
ss_df = pd.DataFrame({'id': submission_id, target_column: submission_oof.reshape(-1)})

In [19]:
wgt = np.array([1.0])

for w in wgts:
    wgt = wgt * (1 - w)
    wgt = np.concatenate([wgt, np.array([w])])

In [20]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "exp_oof_path": experiment_config["experiment"]["exp_oof_path"],
    "use_negative_wgt": USE_NEGATIVE_WGT,
    "max_iter": MAX_ITER,
    "primary_metric": {
        "name": "auc",
        "value": round(scores[-1], 5)
    },
    "models": models,
    "weights": wgt.tolist(),
    "oof_scores": scores,
    "validation": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [21]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

model_weight_df.to_csv(experiment_path / "weights.csv", index=False)

oof_df.to_csv(experiment_path / "oof.csv", index=False)
ss_df.to_csv(experiment_path / f"submission.csv", index=False)